In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [24]:
#load NASA dataset 
kaduna = pd.read_csv(r"C:\AgroGuard AI\data\raw\Kaduna.csv", skiprows=15)
ogun = pd.read_csv(r"C:\AgroGuard AI\data\raw\Ogun.csv", skiprows=15)
benue = pd.read_csv(r"C:\AgroGuard AI\data\raw\Benue.csv", skiprows=15)


#check the first few rows of the kaduna dataset
kaduna.head(5)



,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN,WS2M
0,2015,1,20.94,30.64,13.95,0.0,31.31,20.01,3.95
1,2015,2,20.16,30.33,12.57,0.0,30.25,20.80,4.36
2,2015,3,19.31,29.56,11.76,0.0,29.13,21.46,5.32
3,2015,4,18.20,28.06,10.91,0.0,28.36,20.15,5.34
4,2015,5,17.80,27.08,11.29,0.0,29.69,16.27,4.80


In [25]:
#check the shape of kaduna
print(kaduna.shape)


print(kaduna.columns.to_list())

(3653, 9)
['YEAR', 'DOY', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'ALLSKY_SFC_SW_DWN', 'WS2M']


In [26]:
#checking details for all the datasets
for name, df in zip(['Kaduna', 'Ogun', 'Benue'], [kaduna, ogun, benue]):
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.to_list()}")
    print(f"Missing Values:\n{df.isnull().sum()}")
    print(f"Date Range: {df["YEAR"].min()} to {df["YEAR"].max()}")
    print(f"Info:\n{df.info()}")
    print("\n")

Dataset: Kaduna
Shape: (3653, 9)
Columns: ['YEAR', 'DOY', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'ALLSKY_SFC_SW_DWN', 'WS2M']
Missing Values:
YEAR                 0
DOY                  0
T2M                  0
T2M_MAX              0
T2M_MIN              0
PRECTOTCORR          0
RH2M                 0
ALLSKY_SFC_SW_DWN    0
WS2M                 0
dtype: int64
Date Range: 2015 to 2024
<class 'pandas.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   YEAR               3653 non-null   int64  
 1   DOY                3653 non-null   int64  
 2   T2M                3653 non-null   float64
 3   T2M_MAX            3653 non-null   float64
 4   T2M_MIN            3653 non-null   float64
 5   PRECTOTCORR        3653 non-null   float64
 6   RH2M               3653 non-null   float64
 7   ALLSKY_SFC_SW_DWN  3653 non-null   float64
 8   WS2M               3

In [27]:
#add state identifier and proper date column to each dataset 
def prepare_dataset(df, state_name):
    """
    prepare the dataset by adding a state identifier and creating a proper date column
    """

    #convert Year and DOY to a proper date format
    df["date"] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['DOY'].astype(str), format='%Y-%j')

    #add the state identifier
    df["state"] = state_name

    #rename the columns for clarity
    df = df.rename(columns = {
        'T2M': 'temp_avg',
        'T2M_MAX': 'temp_max',
        'T2M_MIN': 'temp_min',
        'PRECTOTCORR': 'rainfall',
        'RH2M': 'humidity',
        'ALLSKY_SFC_SW_DWN': 'solar_radiation',
        'WS2M': 'wind_speed'
    })

    #reorder the columns
    df = df[[
        "date", "YEAR", "DOY", "state", "temp_avg", "temp_max", 
        "temp_min", "rainfall", "humidity", "solar_radiation", "wind_speed"
    ]]

    return df


In [28]:
#apply the preparation function to each dataset
kaduna_prepared = prepare_dataset(kaduna, "Kaduna")
ogun_prepared = prepare_dataset(ogun, "Ogun")
benue_prepared = prepare_dataset(benue, "Benue")


#check the first few rows of the prepared kaduna dataset
kaduna_prepared.head(5)

,date,YEAR,DOY,state,temp_avg,temp_max,temp_min,rainfall,humidity,solar_radiation,wind_speed
0,2015-01-01,2015,1,Kaduna,20.94,30.64,13.95,0.0,31.31,20.01,3.95
1,2015-01-02,2015,2,Kaduna,20.16,30.33,12.57,0.0,30.25,20.80,4.36
2,2015-01-03,2015,3,Kaduna,19.31,29.56,11.76,0.0,29.13,21.46,5.32
3,2015-01-04,2015,4,Kaduna,18.20,28.06,10.91,0.0,28.36,20.15,5.34
4,2015-01-05,2015,5,Kaduna,17.80,27.08,11.29,0.0,29.69,16.27,4.80


In [29]:
#merge the prepared datasets into a single dataframe
weather_df = pd.concat([kaduna_prepared, ogun_prepared, benue_prepared], ignore_index=True)

#check the shape and columns of the merged dataframe
print("Combined shape:", weather_df.shape)
print("\nStates present:", weather_df['state'].unique())
print("\nDate range:", weather_df['date'].min(), "to", weather_df['date'].max())

Combined shape: (10959, 11)

States present: <ArrowStringArray>
['Kaduna', 'Ogun', 'Benue']
Length: 3, dtype: str

Date range: 2015-01-01 00:00:00 to 2024-12-31 00:00:00


In [30]:
#Engineering new features

def add_engineered_features(df):
    """
    add engineered features to make the model more accurate
    """

    #sort by date and state first 
    df = df.sort_values(["state", "date"]).reset_index(drop=True)

    #flag extreme heat days (temp_max > 36)
    df["heat_stress"] = (df["temp_max"] > 36).astype(int)

    #rolling average rainfall - 7 day window
    df["rainfall_7day_avg"] = df.groupby("state")["rainfall"].transform(
        lambda x: x.rolling(window=7, min_periods=1).mean())
    
    #consecutive dry days (rainfall == 0)
    def consecutive_dry_days(x):
        count = 0
        dry_days = []
        for value in x:
            if value == 0:
                count += 1
            else:
                count = 0
            dry_days.append(count)
        return dry_days
    
    df["consecutive_dry_days"] = df.groupby("state")["rainfall"].transform(
        lambda x: consecutive_dry_days(x))
    
    #consecutive hot days 
    def consecutive_hot_days(x):
        count = 0
        hot_days = []
        for value in x:
            if value > 36:
                count += 1
            else:
                count = 0
            hot_days.append(count)
        return hot_days
    
    df["consecutive_hot_days"] = df.groupby("state")["temp_max"].transform(lambda x: consecutive_hot_days(x))

    #temperature range - weather instability indicator
    df["temp_range"] = df["temp_max"] - df["temp_min"]

    #month column for seasonality
    df["month"] = df["date"].dt.month

    return df



In [31]:
#apply the feature engineering function to the combined dataframe
weather_df = add_engineered_features(weather_df)

#check the first few rows of the final dataframe with engineered features
weather_df.head()

,date,YEAR,DOY,state,temp_avg,temp_max,temp_min,rainfall,humidity,solar_radiation,wind_speed,heat_stress,rainfall_7day_avg,consecutive_dry_days,consecutive_hot_days,temp_range,month
0,2015-01-01,2015,1,Benue,25.88,34.28,18.35,0.0,39.40,21.11,2.30,0,0.0,1,0,15.93,1
1,2015-01-02,2015,2,Benue,24.06,33.75,15.99,0.0,39.66,22.30,3.08,0,0.0,2,0,17.76,1
2,2015-01-03,2015,3,Benue,23.49,32.06,16.87,0.0,35.63,17.92,4.04,0,0.0,3,0,15.19,1
3,2015-01-04,2015,4,Benue,23.06,31.34,17.29,0.0,29.73,20.46,4.39,0,0.0,4,0,14.05,1
4,2015-01-05,2015,5,Benue,21.94,29.39,16.53,0.0,29.82,18.19,4.13,0,0.0,5,0,12.86,1


In [32]:
#check
weather_df.head(30)

,date,YEAR,DOY,state,temp_avg,temp_max,temp_min,rainfall,humidity,solar_radiation,wind_speed,heat_stress,rainfall_7day_avg,consecutive_dry_days,consecutive_hot_days,temp_range,month
0,2015-01-01,2015,1,Benue,25.88,34.28,18.35,0.00,39.40,21.11,2.30,0,0.000000,1,0,15.93,1
1,2015-01-02,2015,2,Benue,24.06,33.75,15.99,0.00,39.66,22.30,3.08,0,0.000000,2,0,17.76,1
2,2015-01-03,2015,3,Benue,23.49,32.06,16.87,0.00,35.63,17.92,4.04,0,0.000000,3,0,15.19,1
3,2015-01-04,2015,4,Benue,23.06,31.34,17.29,0.00,29.73,20.46,4.39,0,0.000000,4,0,14.05,1
4,2015-01-05,2015,5,Benue,21.94,29.39,16.53,0.00,29.82,18.19,4.13,0,0.000000,5,0,12.86,1
5,2015-01-06,2015,6,Benue,21.59,28.58,15.42,0.00,34.57,16.21,3.85,0,0.000000,6,0,13.16,1
6,2015-01-07,2015,7,Benue,22.50,30.65,16.54,0.00,33.24,18.20,4.29,0,0.000000,7,0,14.11,1
7,2015-01-08,2015,8,Benue,22.67,31.26,16.69,0.00,33.27,20.22,3.86,0,0.000000,8,0,14.57,1
8,2015-01-09,2015,9,Benue,21.95,30.58,15.64,0.00,29.65,20.69,4.85,0,0.000000,9,0,14.94,1
9,2015-01-10,2015,10,Benue,21.24,29.44,15.49,0.00,26.11,21.76,5.34,0,0.000000,10,0,13.95,1


In [33]:
#create risk label for the algorithm
# def create_risk_label(df):
#     """
#     create a risk label based on the engineered features
#     """
#     conditions = []

#     for _, row in df.iterrows():
#         #high risk conditions
#         if (row['temp_max'] > 36 or           # Extreme heat
#             row['rainfall'] > 50 or            # Heavy rainfall
#             row['consecutive_dry_days'] > 14 or # Prolonged drought
#             row['consecutive_hot_days'] > 5):   # Extended heat wave
#             conditions.append(2)  # High risk
        
#         #medium risk conditions
#         elif (row['temp_max'] > 32 or           # Warm temperatures
#               row['rainfall'] > 20 or            # Moderate rainfall
#               row['consecutive_dry_days'] > 7 or  # Short drought
#               row['consecutive_hot_days'] > 2 or 
#               row['humidity'] > 85):   # Short heat wave
#             conditions.append(1)  # Medium risk
        
#         else:
#             conditions.append(0)  # Low risk

#     df["risk_label"] = conditions
#     return df

In [ ]:
def create_risk_label(df):
    """
    Create risk label based on combinations of conditions
    rather than direct thresholds. 
    """
    df = df.copy()
    
    # Calculate a continuous risk score first
    risk_score = pd.Series(0.0, index=df.index)
    
    # Temperature contribution (weighted)
    risk_score += (df['temp_max'] - 30).clip(lower=0) * 0.3
    
    # Rainfall extremes (both too much and too little are risky)
    risk_score += (df['rainfall'] - 30).clip(lower=0) * 0.2
    risk_score += df['consecutive_dry_days'] * 0.25
    
    # Humidity extremes
    risk_score += (df['humidity'] - 80).clip(lower=0) * 0.1
    
    # Compound effect — heat AND drought together is worse
    compound = ((df['temp_max'] > 33) & 
                (df['consecutive_dry_days'] > 5)).astype(int)
    risk_score += compound * 3
    
    # Normalize to 0-10 scale
    risk_score = (risk_score - risk_score.min()) / \
                 (risk_score.max() - risk_score.min()) * 10
    
    # Convert to three categories using percentiles
    # This ensures a natural distribution rather than fixed rules
    low_threshold = risk_score.quantile(0.33)
    high_threshold = risk_score.quantile(0.67)
    
    df['risk_label'] = pd.cut(
        risk_score,
        bins=[-np.inf, low_threshold, high_threshold, np.inf],
        labels=[0, 1, 2]
    ).astype(int)
    
    return df

weather_df = create_risk_label(weather_df)

# Check new distribution
print("New risk label distribution:")
print(weather_df['risk_label'].value_counts())
print("\nAs percentage:")
print(weather_df['risk_label'].value_counts(normalize=True) * 100)

New risk label distribution:
risk_label
1    3727
0    3618
2    3614
Name: count, dtype: int64

As percentage:
risk_label
1    34.008577
0    33.013961
2    32.977461
Name: proportion, dtype: float64


In [35]:
weather_df = create_risk_label(weather_df)

# Check distribution of risk labels
print("Risk label distribution:")
print(weather_df['risk_label'].value_counts())
print("\nAs percentage:")
print(weather_df['risk_label'].value_counts(normalize=True) * 100)

Risk label distribution:
risk_label
1    3727
0    3618
2    3614
Name: count, dtype: int64

As percentage:
risk_label
1    34.008577
0    33.013961
2    32.977461
Name: proportion, dtype: float64


In [36]:
#save the processed dataframe to a new CSV file
weather_df.to_csv(r"C:\AgroGuard AI\data\processed\weather_data_processed.csv", index=False)
print("Saved Successfully!")
print("Final shape of the dataset:", weather_df.shape)

Saved Successfully!
Final shape of the dataset: (10959, 18)
